In [ ]:
import os
import cv2 as cv 
import shutil as sh
import numpy as np

In [ ]:
# filting and cleaning 
train_videos_path= r"D:\Python coding\Depi_project\ucsdpeds\vidf\Train_set"
videos_list_train= os.listdir(train_videos_path)
for i in range(len(videos_list_train)):
    videos_frames=os.path.join(train_videos_path,videos_list_train[i])
    frames=os.listdir(videos_frames) 
    if len(frames)<100:
        print("removing")
        sh.rmtree(videos_frames)
    else:
        continue
print('*'*20)
test_videos_path= r"D:\Python coding\Depi_project\ucsdpeds\vidf\Test_set"
videos_list_test= os.listdir(test_videos_path)
for i in range(len(videos_list_test)):
    videos_frames=os.path.join(test_videos_path,videos_list_test[i])
    frames=os.listdir(videos_frames) 
    if len(frames)<100:
        print(videos_list_test[i])
        print("removing")
        sh.rmtree(videos_frames)
    else:
        continue
print('cleaned')
print(f'Trains videos:{len(videos_list_train)} clips')
print(f'Test videos:{len(videos_list_test)} clips')


In [ ]:
#preprocessing (resizing, normalizing, converting to grayscale) # 
IMG_HEIGHT = 128
IMG_WIDTH = 128

def preprocess_clip_frames(clip_path):
    frame_files = sorted([f for f in os.listdir(clip_path) if f.endswith(".png")])

    frames = []

    for frame_file in frame_files:
        frame_path = os.path.join(clip_path, frame_file)

        img = cv.imread(frame_path)

        if img is None:
            continue

        img = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
        img = cv.resize(img, (IMG_WIDTH, IMG_HEIGHT))
        img = img.astype(np.float32) / 255.0

        frames.append(img)

    return np.array(frames, dtype=np.float32)
# motion map
def compute_motion_maps(frames_array):
    motion_maps = []

    for i in range(1, len(frames_array)):
        diff = np.abs(frames_array[i] - frames_array[i-1])
        motion_maps.append(diff)

    motion_maps = np.array(motion_maps, dtype=np.float32)
    motion_maps = np.expand_dims(motion_maps, axis=-1)

    return motion_maps


In [ ]:


all_preprocessed_clips = []
clip_names = sorted(os.listdir(train_videos_path))

for clip  in clip_names:
    clip_path = os.path.join(train_videos_path, clip)

    if os.path.isdir(clip_path):
        frames_array = preprocess_clip_frames(clip_path)
        all_preprocessed_clips.append(frames_array)

all_motion_map_clips = []

for i, frames_array in enumerate(all_preprocessed_clips):
    motion_maps = compute_motion_maps(frames_array)
    all_motion_map_clips.append(motion_maps)

print("*" * 20)
print("Number of processed clips:", len(all_preprocessed_clips))
print("Number of motion-map clips:", len(all_motion_map_clips))

In [ ]:
import matplotlib.pyplot as plt

motion_maps = all_motion_map_clips[6]
for i in range(1,20):
    plt.imshow(motion_maps[i].squeeze(), cmap='gray')
    plt.title(f"Motion Map - Clip 6 - Frame {i+1}")
    plt.axis("off")
    plt.show()


In [ ]:
print("shape:", motion_maps.shape)
print("min:", motion_maps.min())
print("max:", motion_maps.max())
print("mean:", motion_maps.mean())

In [ ]:
# creating sequences of motion maps for LSTM input
SEQUENCE_LENGTH = 10   # 9 motion maps as input, 1 motion map as target

def create_sequences_from_clip(motion_maps, sequence_length=10):
    X = []
    y = []

    for i in range(len(motion_maps) - sequence_length + 1):
        seq = motion_maps[i : i + sequence_length]

        X.append(seq[:sequence_length - 1])   # first 9 motion maps
        y.append(seq[sequence_length - 1])    # 10th motion map

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

all_X = []
all_y = []

for motion_maps in all_motion_map_clips:
    x_clip, y_clip = create_sequences_from_clip(motion_maps, SEQUENCE_LENGTH)
    all_X.append(x_clip)
    all_y.append(y_clip)

all_X = np.concatenate(all_X, axis=0)
all_y = np.concatenate(all_y, axis=0)

print("X shape:", all_X.shape)  
print("y shape:", all_y.shape)  